# CONEXION CON EL STORAGE

In [ ]:
storage_account = "name"
access_key = "key"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

# LECTURA DEL "joined"

In [ ]:
ruta_silver = f"abfss://datalake@{storage_account}.dfs.core.windows.net/Silver/joined"

df_joined = (
    spark.read
        .format("delta")
        .load(ruta_silver)
)

display(df_joined)

# CALIDAD DE DATOS

In [ ]:
from pyspark.sql.functions import col, when, sum

df_calidad = df_silver
    .withColumn(
        "valida_id",
        when(col("id_envio").isNotNull(), 1).otherwise(0)
    )

    .withColumn(
        "valida_peso",
        when(col("peso_kg") > 0, 1).otherwise(0)
    )
    .withColumn(
        "valida_estado",
        when(col("estado").isin(
                "Entregado",
                "En tránsito",
                "Demorado",
                "Cancelado"
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "valida_distancia",
        when(col("distancia_km") > 0, 1).otherwise(0)
    )

In [ ]:
df_calidad.select(
    sum(when(col("valida_id") == 0, 1).otherwise(0)).alias("ids_invalidos"),
    sum(when(col("valida_peso") == 0, 1).otherwise(0)).alias("pesos_invalidos"),
    sum(when(col("valida_estado") == 0, 1).otherwise(0)).alias("estados_invalidos"),
    sum(when(col("valida_distancia") == 0, 1).otherwise(0)).alias("distancias_invalidas")
).show()


In [ ]:
df envios validos = df calidad.filter(
    (col("valida_id") == 1) &
    (col("valida_peso") == 1) &
    (col("valida_estado") == 1) &
    (col("valida_distancia") == 1)
)

In [ ]:
print("Registros originales:", df_silver.count())
print("Registros válidos:", df_envios_validos.count())
print("Registros rechazados:", df_silver.count() - df_envios_validos.count())


In [ ]:
df_calidad.filter(
    (col("valida_id") == 0) 
    (col("valida_peso") == 0) 
    (col("valida_estado") == 0)
    (col("valida_distancia") == 0)
).select(
    "id_envio",
    "peso_kg",
    "estado",
    "distancia_km",
    "valida_id",
    "valida_peso",
    "valida_estado",
    "valida_distancia"
).show()


In [ ]:
df_silver = df_envios_validos.drop(
    "valida_id",
    "valida_peso",
    "valida_estado",
    "valida distancia"
)

print("Registros finales para Gold:", df_silver.count())


# PRIMERA AGREGACIÓN

In [ ]:
from pyspark.sql.functions import count, when, col, avg, round

df_envios_sucursal = (
    df_silver
    .groupBy("sucursal")
    .agg(
        count("*").alias("total_envios"),
        count(when(col("estado") == "Entregado", 1)).alias("entregados")
        count(when(col("estado") == "En Tránsito", 1)).alias("en_transito"),
        count(when(col("estado") == "Demorado", 1)).alias("demorados"),
        count(when(col("estado") == "Cancelado", 1)).alias("cancelados"),
        round(avg("peso_kg"),2).alias("peso_promedio_kg"),
        round(avg("distancias_km"),2).alias("distancia_promedio_km")
    )
)

display(df_envios_sucursal)


In [ ]:
ruta_gold_envios = f"abfss://datalake@{storage_account}.dfs.core.windows.net/Gold/envios_por_sucursal"

df_envios_sucursal.write
    .format("delta")
    .mode("overwrite")
    .save(ruta_gold_envios)

In [ ]:
#Comprobación
dbutils.fs.ls(ruta_gold_envios)

# SEGUNDA AGREGACIÓN

In [ ]:
from pyspark.sql.functions import (count,sum,avg,when,round,col)

df_kpi_envios = df_silver.groupBy("sucursal").agg(
    count("id_envio").alias("total_envios"),
    sum(when(col("estado") == "Entregado", 1).otherwise(0)).alias("envios_entregados"),
    sum(when(col("estado") == "En tránsito", 1).otherwise(0)).alias("envios_en_transito"),
    sum(when(col("estado") == "Demorado", 1).otherwise(0)).alias("envios_demorados"),
    sum(when(col("estado") == "Cancelados", 1).otherwise(0)).alias("envios_cancelados"),
    round(avg("distancia_km"), 2).alias("distancia_promedio_km"),
    round(avg("peso_kg"), 2).alias("peso_promedio_kg"),
    round(sum("distancia_km"), 2).alias("distancia_total_km"),
    round(avg("duracion_min), 2).alias("duracion_promedio_min"),
    round(sum("costo_combustible_estimado),2).alias("costo_combustible_total"),
    round(avg("costo_combustible_estimado),2).alias("costo_combustible_promedio"),
    sum("peajes").alias("peajes_totales")
)

df_kpi_envios = df_kpi_envios.withColumn(
    "porcentaje_entregados",
    round(
        (col("envios_entregados") / col("total_envios")) *100,
        2
    )
)

df_kpi_envios = df_kpi_envios.orderBy("sucursal")

df_kpi_envios.show()
df_kpi_envios.printSchema()

# TERCERA AGREGACIÓN

In [ ]:
from pyspark.sql.functions import (count,sum,avg,when,round,col)

df_rutas = df_silver.groupBy("origen", "destino").agg(
    count("id_envio").alias("cantidad_envios"),
    round(avg("distancia_km"),2).alias("disancia_promedio_km"),
    round(avg("duracion_min"),2).alias("duracion_promedio_min"),
    round(sum("costo_combustible_estimado"),2).alias("costo_combustible_total"),
    round(avg("costo_combustible_estimado"),2).alias("costo_combustible_promedio"),
    sum("peajes").alias("peajes_totales"),
    round(avg("peso_kg"),2 ).alias("peso_promedio_kg")
)

df_rutas = df_rutas.orderBy(col("cantidad_envios").desc())

df_rutas.show()